In [ ]:
import pandas as pd
import numpy as np
import string
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity


Calculate the frequency distribution of the correct answer (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?
*

In [ ]:
train = pd.read_csv(""../data/train.csv"")
freq = train["answer"].value_counts()

q1 = freq.max() + freq.min()
q1

After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?

In [ ]:
translator = str.maketrans('', '', string.punctuation)
#The str.maketrans() method in Python creates a translation table (a dictionary-like mapping)
#that specifies how characters in a string should be replaced, mapped, or deleted. 
def clean_prompt(text):
    text = str(text).lower()
    text = text.translate(translator)
    return text

vocab = set()

for text in train["prompt"]:
    words = clean_prompt(text).split()
    vocab.update(words)

q2 = len(vocab)
q2

Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?

In [ ]:
row1 = train.iloc[0]
words = clean_prompt(row1["prompt"]).split()
filtered = [w for w in words if w not in ENGLISH_STOP_WORDS]

q3 = len(filtered)
q3


Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?



In [ ]:
combined_docs = []

for _, row in train.iterrows():
    combined = " ".join([str(row["prompt"]),str(row["A"]),str(row["B"]),str(row["C"]),
        str(row["D"]),str(row["E"])])
    combined_docs.append(combined)
vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(combined_docs)

q4 = len(vectorizer.get_feature_names_out())
q4

Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).

In [ ]:
prompt_vec = vectorizer.transform([row1["prompt"]])
a_vec = vectorizer.transform([row1["A"]])

sim = cosine_similarity(prompt_vec, a_vec)[0][0]

q5 = round(sim, 4)
q5

Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options . Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.
*

In [ ]:
correct = 0
for _, row in train.iterrows():

    p = vectorizer.transform([row["prompt"]])

    scores = {}

    for opt in ["A","B","C","D","E"]:
        o = vectorizer.transform([row[opt]])
        scores[opt] = cosine_similarity(p, o)[0][0]

    pred = max(scores, key=scores.get)

    if pred == row["answer"]:
        correct += 1

q6 = 100 * correct / len(train)
q6

If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?
*

In [ ]:
# Ground truth C
# Prediction C A B
q7 = 1.0
q7


If the ground truth answer for a question is B, what is the MAP@3 score if a model predicts D B E?

In [ ]:
# Ground truth B
# Prediction D B E
q8 = 1/2
q8

The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?


In [ ]:
freq_order = freq.index.tolist()

top3 = freq_order[:3]

def apk(actual, pred):
    for i,p in enumerate(pred[:3]):
        if p == actual:
            return 1/(i+1)
    return 0

scores = [
    apk(ans, top3)
    for ans in train["answer"]
]

q9 = np.mean(scores)
q9

The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?

In [ ]:
scores = []

for _, row in train.iterrows():

    p = vectorizer.transform([row["prompt"]])

    sims = []

    for opt in ["A","B","C","D","E"]:
        o = vectorizer.transform([row[opt]])
        sims.append(
            (opt, cosine_similarity(p,o)[0][0])
        )

    sims.sort(key=lambda x:x[1], reverse=True)

    pred = [x[0] for x in sims[:3]]

    scores.append(
        apk(row["answer"], pred)
    )

q10 = np.mean(scores)
q10